# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples.

- [View dataset and Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")


## 2. Data Overview
Explore the available record sets and their respective fields. For each entity, we'll reference them by their `@id` field as required by the `mlcroissant` API.


In [ ]:
# List the available record sets by @id with their details
print('Available record sets:')
record_sets = list(dataset.record_sets)
if not record_sets:
    print('No record sets declared at top-level metadata. Attempting to parse from distribution/file objects...')
    # Fallback: infer record set ids from files with tabular data
    # The distribution entries contain information about possible tabular data
    if hasattr(metadata, 'distribution'):
        for dl in metadata.distribution:
            rec_set_id = dl['@id'] if isinstance(dl, dict) and '@id' in dl else str(dl)
            print(f'  - {rec_set_id}')
    else:
        print('No record sets or tabular distribution found!')
else:
    for rs in record_sets:
        print(f"  - {rs['@id']}: {rs.get('name','')} | {rs.get('description','')}")


### List Fields for a Record Set
For demonstration, we use the main record set file under distribution (as per dataset, there is likely only one primary table), referenced by its `@id`.


In [ ]:
# Identify main record set from distribution
if hasattr(metadata, 'distribution'):
    main_record_set_id = metadata.distribution[0]['@id'] if isinstance(metadata.distribution[0], dict) else metadata.distribution[0]
    print(f"Using record set: {main_record_set_id}")
else:
    raise ValueError("No distribution (record set) found in metadata.")

print('\nListing fields and columns (@id and description):')
record_set_obj = None
for rs in dataset.record_sets:
    if rs['@id'] == main_record_set_id:
        record_set_obj = rs
        break

if record_set_obj and 'field' in record_set_obj:
    # Many Croissant schemas have a 'field' key listing field dicts
    for fld in record_set_obj['field']:
        if isinstance(fld, dict):
            print(f"  - {fld.get('@id', fld)}: {fld.get('name','')} ({fld.get('description','')})")
        else:
            print(f"  - {fld}")
else:
    print("Fields not explicitly listed in the record set definition.")


## 3. Data Extraction
Load the data from a specific record set into a pandas DataFrame for analysis.
We use the record set and field `@id`s discovered above for extraction.


In [ ]:
# Prepare list of record set ids
record_sets_ids = [main_record_set_id]
dataframes = {}

for record_set_id in record_sets_ids:
    # Each record is a mapping from field @id -> value
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set '{record_set_id}' with shape {df.shape}")

print(f"\nColumns in main record set DataFrame ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze a numeric field in the protein data. We will filter on a numeric column, normalize it, and group by another key field.

For demonstration, let's use the molecular weight field (`mw` or similar, referenced by its column `@id`) and group by description, accession, or similar identifier. (You'll want to check the output above for the exact column `@id` that corresponds to molecular weight.)

In [ ]:
# Picking typical candidate field names/IDs for demonstration. Replace with your actual field @ids as appropriate.
numeric_field = None
# Try the most common field ids for molecular weight
for candidate in ['mw', 'MW', 'molecular_weight', 'schema:molecularWeight', 'cr:mw_kda', 'MW_kDa', 'mw_kDa']:
    if candidate in dataframes[main_record_set_id].columns:
        numeric_field = candidate
        break

if numeric_field is None:
    # Fallback: just pick the first float/integer column
    sample_row = dataframes[main_record_set_id].iloc[0]
    for col in dataframes[main_record_set_id].columns:
        if pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][col]):
            numeric_field = col
            break

assert numeric_field is not None, "No numeric field found for EDA."
print(f"Using numeric field for EDA: {numeric_field}")

threshold = dataframes[main_record_set_id][numeric_field].quantile(0.75)  # Use 75th percentile as example
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df[[numeric_field]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a likely categorical field, e.g. 'description' or 'accession'
group_field = None
for candidate in ['description', 'Description', 'name', 'accession', 'id', 'cr:accession', 'cr:description', 'protein_id', 'uniprot_accession']:
    if candidate in dataframes[main_record_set_id].columns and candidate != numeric_field:
        group_field = candidate
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"{numeric_field}_mean")
    print(f"\nGrouped data by {group_field} (average {numeric_field}):")
    print(grouped_df.head())
else:
    print('No suitable field found for grouping.')

## 5. Visualization
Visualize the distribution of the selected numeric field after filtering. We'll use matplotlib and seaborn for a histogram and boxplot.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(16, 6))
plt.subplot(1,2,1)
sns.histplot(filtered_df[numeric_field], kde=True)
plt.title(f'Distribution of {numeric_field} (Filtered)')

plt.subplot(1,2,2)
sns.boxplot(x=filtered_df[numeric_field])
plt.title(f'Boxplot of {numeric_field} (Filtered)')
plt.show()

## 6. Conclusion

We've loaded and explored a FAIR^2 dataset describing mass spectrometry measurements of extracellular vesicles from human mast cells. We parsed metadata, loaded tabular data by record set `@id`, examined available fields, selected a numeric measurement for normalization and grouped analysis, and visualized the resulting distributions.

- This approach can be extended to analyze other fields, filter by categorical properties, or integrate with protein databases for further annotation.
- All references to dataset entities (record sets, fields) were made using their Croissant schema `@id`, following best practices for machine-actionable metadata.
